# Titanic Survival Prediction

## Introduction

The sinking of the RMS Titanic is one of the most infamous maritime disasters in history. On April 15, 1912, the Titanic sank after colliding with an iceberg, resulting in the deaths of 1,502 out of 2,224 passengers and crew.

In this notebook, we will walk through the complete machine learning workflow to predict which passengers survived the disaster. This dataset is a classic benchmark in data science and provides an excellent opportunity to apply the key steps of the ML pipeline:

1. **Data Preparation & Exploratory Data Analysis (EDA)**
2. **Feature Engineering**
3. **Model Training & Hyperparameter Tuning**
4. **Evaluation & Model Selection**

We will leverage **scikit-learn** for preprocessing, modeling, and evaluation throughout this notebook.

In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import set_config
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Configure scikit-learn output
set_config(display='diagram')

---

## 1. Data Preparation & Exploratory Data Analysis (EDA)

We begin by loading the training data and examining its structure. The goal is to understand the features, their types, and any data quality issues such as missing values.

In [ ]:
# Load training data
train = pd.read_csv('Datasets/train.csv')

print(f"Dataset shape: {train.shape}")
print(f"\n{'='*60}")
print("Dataset Info:")
print(f"{'='*60}")
train.info()

### 1.1 Initial Overview

The dataset contains 891 passengers with 12 features. Key observations:
- **Target variable**: `Survived` (binary: 1 = survived, 0 = did not survive)
- **PassengerId** and **Name** are identifiers — not useful for prediction
- **Pclass**, **Sex**, **SibSp**, **Parch**, **Embarked** are categorical
- **Age**, **Fare** are numerical
- **Cabin** has significant missing values
- **Ticket** is a mix of alphanumeric codes

In [ ]:
# First few rows
train.head(10)

In [ ]:
# Missing values summary
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(1)
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print("Missing Values:")
missing_df

### 1.2 Data Type Verification

As the assignment guidelines emphasize, we must ensure columns have the correct data types. A column containing integers may not be truly numerical, and string columns may actually represent numerical values.

In [ ]:
# Check data types and sample values
for col in train.columns:
    print(f"{col}: {train[col].dtype} — Sample: {train[col].dropna().iloc[:3].tolist()}")

### 1.3 Univariate Analysis — Categorical Variables

For categorical variables, we examine the unique values, their counts, and the mode (most frequent class).

In [ ]:
# Categorical variables analysis
cat_cols = ['Sex', 'Embarked', 'Pclass', 'SibSp', 'Parch']

for col in cat_cols:
    print(f"\n{col} — Unique values: {train[col].nunique()}")
    print(f"Mode: {train[col].mode().iloc[0]}")
    print(train[col].value_counts())

In [ ]:
# Visualize categorical variables
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(data=train, x=col, hue='Survived', ax=axes[i])
    axes[i].set_title(f'{col} vs Survival')

# Survived distribution
axes[-1].pie(train['Survived'].value_counts(), labels=['Not Survived', 'Survived'], 
             autopct='%1.1f%%', colors=['#ff6b6b', '#51cf66'], startangle=90)
axes[-1].set_title('Survival Distribution')

plt.tight_layout()
plt.savefig('plots/categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.4 Univariate Analysis — Numerical Variables

For numerical variables, we use descriptive statistics and histograms to understand distributions, ranges, and potential outliers.

In [ ]:
# Numerical variables statistics
num_cols = ['Age', 'Fare']
train[num_cols].describe()

In [ ]:
# Visualize numerical variables
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(num_cols):
    sns.histplot(data=train, x=col, hue='Survived', kde=True, bins=30, ax=axes[i])
    axes[i].set_title(f'{col} Distribution by Survival')

plt.tight_layout()
plt.savefig('plots/numerical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.5 Bivariate Analysis — Correlations

We examine relationships between variables and with the target. For numerical variables, we compute a correlation matrix. We want low correlation between independent variables and high correlation with the target.

In [ ]:
# Correlation matrix for numerical features
num_cols_with_target = ['Survived', 'Age', 'Fare', 'Pclass', 'SibSp', 'Parch']
corr_matrix = train[num_cols_with_target].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            linewidths=0.5, square=True)
plt.title('Correlation Matrix — Numerical Features')
plt.tight_layout()
plt.savefig('plots/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCorrelation with Survived:")
print(corr_matrix['Survived'].sort_values(ascending=False))

### 1.6 Key Findings from EDA

Based on our exploratory analysis:

- **Sex** is a strong predictor: women had significantly higher survival rates
- **Pclass** matters: higher class passengers (lower Pclass number) survived more often
- **Age** shows that children were prioritized; there is a peak in survival for young ages
- **Fare** is positively correlated with survival (linked to Pclass)
- **Embarked** shows some variation by port of departure
- **Cabin** has ~77% missing values — may need special handling or could be dropped

Next, we move to feature engineering to prepare the data for modeling.

### 1.7 Test Set Exploration

As mentioned in the assignment guidelines, the dataset has already been split into training and test sets. We must explore the test set separately to understand its distribution and ensure our preprocessing pipeline will handle it correctly. Importantly, we set aside the test data *before* feature engineering — the test data must not influence our preprocessing decisions.

In [ ]:
# Load test data
test = pd.read_csv('Datasets/test.csv')

print(f"Test set shape: {test.shape}")
print(f"\n{'='*60}")
test.info()

In [ ]:
# Missing values in test set
missing_test = test.isnull().sum()
missing_test_pct = (missing_test / len(test) * 100).round(1)
missing_test_df = pd.DataFrame({'Missing Count': missing_test, 'Percentage (%)': missing_test_pct})
missing_test_df = missing_test_df[missing_test_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print("Missing Values in Test Set:")
missing_test_df

In [ ]:
# Compare distributions between train and test
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['Pclass', 'Sex', 'Embarked']):
    train_counts = train[col].value_counts(normalize=True).sort_index()
    test_counts = test[col].value_counts(normalize=True).sort_index()
    
    comparison = pd.DataFrame({'Train': train_counts, 'Test': test_counts})
    comparison.plot(kind='bar', ax=axes[i], color=['#4c78a8', '#f58518'])
    axes[i].set_title(f'{col} Distribution: Train vs Test')
    axes[i].set_ylabel('Proportion')
    axes[i].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('plots/train_test_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare numerical distributions
print("Numerical Features — Train vs Test Statistics:\n")
print("Age:")
print(f"  Train: mean={train['Age'].mean():.1f}, median={train['Age'].median():.1f}, std={train['Age'].std():.1f}")
print(f"  Test:  mean={test['Age'].mean():.1f}, median={test['Age'].median():.1f}, std={test['Age'].std():.1f}")
print("\nFare:")
print(f"  Train: mean={train['Fare'].mean():.1f}, median={train['Fare'].median():.1f}, max={train['Fare'].max():.1f}")
print(f"  Test:  mean={test['Fare'].mean():.1f}, median={test['Fare'].median():.1f}, max={test['Fare'].max():.1f}")

### 1.8 Key Observations from Test Set

- The test set has **418 passengers** (no `Survived` column — this is what we need to predict)
- **Cabin** is also heavily missing in the test set
- **Age** and **Fare** distributions are similar between train and test
- **Pclass**, **Sex**, and **Embarked** distributions appear comparable

These observations confirm that our preprocessing strategy developed on the training set should transfer reasonably well to the test set. We now proceed with feature engineering.

## 2. Feature Engineering

We now build a compact, assignment-aligned feature set. The focus is on strong, interpretable signals, careful imputation, and consistent preprocessing fitted on the training data only. We also pay close attention to leakage: the test set is only used for domain-driven feature construction, while all learned preprocessing steps such as imputation, encoding, scaling, and model tuning are fit on the training split only.

### 2.1 Combine Datasets

We concatenate train and test to keep feature engineering consistent. The target `Survived` is stored separately and never used during preprocessing. This combine step is only used for features that do not depend on the target, and we still fit every learned preprocessing step on the training portion only so the test set never influences model selection.

In [ ]:
# Preserve split point and target
n_train = train.shape[0]
y_train = train['Survived'].copy()

# Combine train and test without the target
combined = pd.concat([train.drop(columns=['Survived']), test], ignore_index=True)

print(f'Combined shape: {combined.shape}')
print(f'Train rows: {n_train}, test rows: {len(test)}')
print(f'Columns: {combined.columns.tolist()}')

### 2.2 Extract Titles

Names contain useful social information. We extract titles and collapse rare ones into a single `Rare` class to keep the feature robust on a small dataset.

In [ ]:
# Extract and simplify titles
combined['Title'] = combined['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
title_map = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
    'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
}
combined['Title'] = combined['Title'].replace(title_map)
combined['Title'] = combined['Title'].where(combined['Title'].isin(['Mr', 'Mrs', 'Miss', 'Master']), 'Rare')

print(combined['Title'].value_counts())

In [ ]:
# Visualize titles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
title_order = ['Mr', 'Rare', 'Mrs', 'Miss', 'Master']
sns.countplot(data=combined, x='Title', order=title_order, ax=axes[0])
axes[0].set_title('Title Distribution')
axes[0].tick_params(axis='x', rotation=45)

train_view = combined.iloc[:n_train].copy()
train_view['Survived'] = y_train.values
sns.barplot(data=train_view, x='Title', y='Survived', order=title_order, ax=axes[1])
axes[1].set_title('Survival Rate by Title')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 2.3 Family Features

`FamilySize` captures whether a passenger travelled alone, with a partner, or in a larger group. It is a simple but informative proxy for support during evacuation.

In [ ]:
# FamilySize from SibSp and Parch
combined['FamilySize'] = combined['SibSp'] + combined['Parch'] + 1
print(combined['FamilySize'].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(combined['FamilySize'], discrete=True, bins=range(1, 12), ax=axes[0])
axes[0].set_title('FamilySize Distribution')

sns.barplot(data=train_view.assign(FamilySize=combined.iloc[:n_train]['FamilySize'].values), x='FamilySize', y='Survived', ax=axes[1])
axes[1].set_title('Survival Rate by FamilySize')

plt.tight_layout()
plt.show()

### 2.4 Cabin Availability

`Cabin` is mostly missing, but its presence is informative. We keep a binary `HasCabin` flag and drop the sparse raw cabin text later.

In [ ]:
# Cabin indicator
combined['HasCabin'] = combined['Cabin'].notna().astype(int)
print(combined['HasCabin'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='HasCabin', data=combined, ax=axes[0])
axes[0].set_title('Cabin Availability')

train_view['HasCabin'] = combined.iloc[:n_train]['HasCabin'].values
sns.barplot(data=train_view, x='HasCabin', y='Survived', ax=axes[1])
axes[1].set_title('Survival Rate by HasCabin')

plt.tight_layout()
plt.show()

### 2.5 Age Imputation

Age is imputed with the median per `Title`, computed on the training portion only. This keeps the imputation meaningful while avoiding leakage.

In [ ]:
# Age imputation by Title
age_before = combined['Age'].copy()
age_medians = combined.iloc[:n_train].groupby('Title')['Age'].median()
print(age_medians)

for title, median_age in age_medians.items():
    mask = combined['Age'].isna() & (combined['Title'] == title)
    combined.loc[mask, 'Age'] = median_age

missing_age_after = combined['Age'].isna().sum()
print(f'Missing Age after imputation: {missing_age_after}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(age_before.dropna(), bins=30, kde=True, ax=axes[0])
axes[0].set_title('Age Before Imputation')
sns.histplot(combined['Age'], bins=30, kde=True, ax=axes[1])
axes[1].set_title('Age After Imputation')
plt.tight_layout()
plt.show()

### 2.6 Fare Imputation and Log Transform

Fare is right-skewed, so we impute missing values with the median per `Pclass` and then apply a log transform.

In [ ]:
# Fare imputation by Pclass and log transform
fare_before = combined['Fare'].copy()
fare_medians = combined.iloc[:n_train].groupby('Pclass')['Fare'].median()
print(fare_medians)

for pclass, median_fare in fare_medians.items():
    mask = combined['Fare'].isna() & (combined['Pclass'] == pclass)
    combined.loc[mask, 'Fare'] = median_fare

combined['FareLog'] = np.log1p(combined['Fare'])
missing_fare_after = combined['Fare'].isna().sum()
print(f'Missing Fare after imputation: {missing_fare_after}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(fare_before.dropna(), bins=30, kde=True, ax=axes[0])
axes[0].set_title('Fare Before Imputation')
sns.histplot(combined['FareLog'], bins=30, kde=True, ax=axes[1])
axes[1].set_title('Fare Log-Transformed')
plt.tight_layout()
plt.show()

### 2.7 Baseline: Women and Children First

As a simple benchmark, we predict survival for women and children and non-survival for everyone else. This is a meaningful, domain-driven baseline.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

baseline_frame = combined.iloc[:n_train].copy()
baseline_pred = ((baseline_frame['Sex'] == 'female') | (baseline_frame['Age'] < 16)).astype(int)
acc = accuracy_score(y_train, baseline_pred)
cm = confusion_matrix(y_train, baseline_pred)

print(f'Baseline accuracy: {acc:.3f}')
print(classification_report(y_train, baseline_pred))

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Women and Children First - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### 2.8 Encoding

We use scikit-learn encoders for the categorical variables. `Title` is encoded ordinally, `Sex` is encoded as a binary 0/1 feature, and `Embarked` is reduced to a single binary `Embarked_C` feature, since Cherbourg is the port we want to isolate explicitly.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# Encode Pclass manually so 1st class stays highest
combined['Pclass'] = combined['Pclass'].map({1: 2, 2: 1, 3: 0})

# Ordinal encoding for Title
title_encoder = OrdinalEncoder(
    categories=[['Mr', 'Rare', 'Mrs', 'Miss', 'Master']],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
title_encoder.fit(combined.iloc[:n_train][['Title']])
combined[['Title']] = title_encoder.transform(combined[['Title']])

# One-hot for Sex only, keep male as the explicit signal
combined['Sex'] = combined['Sex'].map({'female': 0, 'male': 1})

# Binary Embarked feature for Cherbourg
combined['Embarked_C'] = (combined['Embarked'] == 'C').astype(int)
combined = combined.drop(columns=['Embarked'])

print(combined.head())

### 2.9 Scaling

We standardize the continuous features used by linear models. The scaler is fit on the training portion only and then applied to the full combined dataset.

In [ ]:
# Scale continuous features
scale_cols = ['Age', 'FareLog', 'FamilySize']
scaler = StandardScaler()
scaler.fit(combined.iloc[:n_train][scale_cols])
combined[scale_cols] = scaler.transform(combined[scale_cols])

print(combined[scale_cols].describe().round(2))

### 2.10 Final Feature Set

We now drop the raw text and unused source columns, split the data back into train/test, and inspect the final feature correlations.

In [ ]:
# Drop raw columns no longer needed
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch', 'Fare']
combined = combined.drop(columns=drop_cols)

# Split back into train and test features
X_train = combined.iloc[:n_train].copy()
X_test = combined.iloc[n_train:].copy()

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(X_train.columns.tolist())

plt.figure(figsize=(10, 8))
sns.heatmap(X_train.corr(numeric_only=True), cmap='coolwarm', center=0)
plt.title('Final Feature Correlation Matrix')
plt.tight_layout()
plt.show()

### 2.11 Summary

Final feature set:

- `Pclass`
- `Title`
- `Age`
- `FareLog`
- `FamilySize`
- `HasCabin`
- `Sex`
- `Embarked_C`

This keeps the notebook compact, interpretable, and aligned with the assignment guidelines.

## 3. Model Training & Hyperparameter Tuning

We now train three classification models on the engineered features and compare them against the baseline from Section 2 (`Women and Children First`). We use 5-fold stratified cross-validation for a stable comparison across models.

### 3.1 Training Setup

We evaluate each model with the same validation strategy and the same metrics so the comparison remains fair. Since the features are already engineered, encoded, and scaled, the models can be trained directly on `X_train`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X = X_train.copy()
y = y_train.copy()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1'}

baseline_pred = ((X['Sex'] == 0) | (X['Age'] < 16)).astype(int)
baseline_metrics = {
    'Model': 'Baseline',
    'Accuracy': accuracy_score(y, baseline_pred),
    'Precision': precision_score(y, baseline_pred, zero_division=0),
    'Recall': recall_score(y, baseline_pred, zero_division=0),
    'F1': f1_score(y, baseline_pred, zero_division=0)
}
baseline_cm = confusion_matrix(y, baseline_pred)
baseline_metrics

In [ ]:
# Baseline confusion matrix
plt.figure(figsize=(5, 4))
sns.heatmap(baseline_cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Baseline: Women and Children First')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### 3.2 Logistic Regression

Logistic Regression is a strong baseline classifier for small tabular datasets. It is simple, interpretable, and often performs surprisingly well when the features are informative.

In [ ]:
logreg = LogisticRegression(max_iter=2000, random_state=42)
logreg_grid = {
    'C': [0.1, 1.0, 10.0],
    'solver': ['liblinear'],
    'penalty': ['l1', 'l2']
}

logreg_search = GridSearchCV(logreg, logreg_grid, cv=cv, scoring='f1', n_jobs=-1, return_train_score=True)
logreg_search.fit(X, y)
logreg_best = logreg_search.best_estimator_
logreg_oof = cross_val_predict(logreg_best, X, y, cv=cv)
logreg_scores = cross_validate(logreg_best, X, y, cv=cv, scoring=scoring, n_jobs=-1)
logreg_metrics = {
    'Model': 'Logistic Regression',
    'Accuracy': logreg_scores['test_accuracy'].mean(),
    'Precision': logreg_scores['test_precision'].mean(),
    'Recall': logreg_scores['test_recall'].mean(),
    'F1': logreg_scores['test_f1'].mean()
}
logreg_cm = confusion_matrix(y, logreg_oof)
print(logreg_search.best_params_)
logreg_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(logreg_cm, annot=True, fmt='d', cmap='Greens', cbar=False, ax=axes[0])
axes[0].set_title('Logistic Regression Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

logreg_cv = pd.DataFrame(logreg_search.cv_results_)
logreg_plot = logreg_cv.pivot_table(index='param_C', columns='param_penalty', values='mean_test_score')
sns.heatmap(logreg_plot, annot=True, fmt='.3f', cmap='viridis', ax=axes[1])
axes[1].set_title('LogReg F1 by C / Penalty')
plt.tight_layout()
plt.show()

### 3.3 Random Forest

Random Forest can capture non-linear relationships and feature interactions without much manual feature engineering. It is a strong candidate for this dataset.

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf_search = GridSearchCV(rf, rf_grid, cv=cv, scoring='f1', n_jobs=-1, return_train_score=True)
rf_search.fit(X, y)
rf_best = rf_search.best_estimator_
rf_oof = cross_val_predict(rf_best, X, y, cv=cv)
rf_scores = cross_validate(rf_best, X, y, cv=cv, scoring=scoring, n_jobs=-1)
rf_metrics = {
    'Model': 'Random Forest',
    'Accuracy': rf_scores['test_accuracy'].mean(),
    'Precision': rf_scores['test_precision'].mean(),
    'Recall': rf_scores['test_recall'].mean(),
    'F1': rf_scores['test_f1'].mean()
}
rf_cm = confusion_matrix(y, rf_oof)
print(rf_search.best_params_)
rf_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Greens', cbar=False, ax=axes[0])
axes[0].set_title('Random Forest Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

rf_cv = pd.DataFrame(rf_search.cv_results_)
rf_plot = rf_cv.pivot_table(index='param_max_depth', columns='param_n_estimators', values='mean_test_score', aggfunc='mean')
sns.heatmap(rf_plot, annot=True, fmt='.3f', cmap='viridis', ax=axes[1])
axes[1].set_title('RF F1 by Depth / Trees')
plt.tight_layout()
plt.show()

### 3.4 Gradient Boosting

Gradient Boosting builds trees sequentially and often performs very well on structured data when tuned carefully. It is a good final candidate for this assignment.

In [ ]:
gb = GradientBoostingClassifier(random_state=42)
gb_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3],
    'subsample': [0.8, 1.0]
}

gb_search = GridSearchCV(gb, gb_grid, cv=cv, scoring='f1', n_jobs=-1, return_train_score=True)
gb_search.fit(X, y)
gb_best = gb_search.best_estimator_
gb_oof = cross_val_predict(gb_best, X, y, cv=cv)
gb_scores = cross_validate(gb_best, X, y, cv=cv, scoring=scoring, n_jobs=-1)
gb_metrics = {
    'Model': 'Gradient Boosting',
    'Accuracy': gb_scores['test_accuracy'].mean(),
    'Precision': gb_scores['test_precision'].mean(),
    'Recall': gb_scores['test_recall'].mean(),
    'F1': gb_scores['test_f1'].mean()
}
gb_cm = confusion_matrix(y, gb_oof)
print(gb_search.best_params_)
gb_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(gb_cm, annot=True, fmt='d', cmap='Greens', cbar=False, ax=axes[0])
axes[0].set_title('Gradient Boosting Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

gb_cv = pd.DataFrame(gb_search.cv_results_)
gb_plot = gb_cv.pivot_table(index='param_learning_rate', columns='param_n_estimators', values='mean_test_score', aggfunc='mean')
sns.heatmap(gb_plot, annot=True, fmt='.3f', cmap='viridis', ax=axes[1])
axes[1].set_title('GB F1 by LR / Trees')
plt.tight_layout()
plt.show()

### 3.5 Model Comparison

We now compare all models against the baseline to see which one improves the most and which one is the best candidate for the final evaluation.

In [ ]:
results = pd.DataFrame([baseline_metrics, logreg_metrics, rf_metrics, gb_metrics])
results['Accuracy Δ vs Baseline'] = results['Accuracy'] - baseline_metrics['Accuracy']
results['Precision Δ vs Baseline'] = results['Precision'] - baseline_metrics['Precision']
results['Recall Δ vs Baseline'] = results['Recall'] - baseline_metrics['Recall']
results['F1 Δ vs Baseline'] = results['F1'] - baseline_metrics['F1']
results = results.set_index('Model')
results.round(3)

In [ ]:
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1']
ax = results[metric_cols].plot(kind='bar', figsize=(12, 5))
ax.set_title('Model Metrics Comparison')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

### 3.6 Conclusion

The best model is the one with the highest cross-validated F1 score, while also improving clearly over the baseline. We use that model in the next section for final evaluation and Kaggle submission.

## 4. Final Model Evaluation

In this section we take the strongest model from Section 3, fit it one final time on the full training set, and inspect how it behaves on the training data through out-of-fold predictions. This gives us a compact but clear view of what the model learned and where it still makes mistakes.

### 4.1 Select Final Model

We choose the model with the highest cross-validated F1 score from Section 3 because F1 balances precision and recall better than accuracy alone on this slightly imbalanced dataset.

In [ ]:
best_model_name = results['F1'].idxmax()
best_model_name

In [ ]:
model_map = {
    'Logistic Regression': logreg_best,
    'Random Forest': rf_best,
    'Gradient Boosting': gb_best,
}

final_model = model_map[best_model_name]
final_model

### 4.2 Retrain on Full Training Set

After we have selected the best model, we fit it again on the entire training set. This is the version we would carry forward into the final prediction stage, because it uses all available labeled data.

In [ ]:
X_full = X_train.copy()
y_full = y_train.copy()

final_model.fit(X_full, y_full)
print(f'Final model: {best_model_name}')

### 4.3 Compact Error Analysis

Instead of evaluating on the same data the model was just trained on, we use the out-of-fold predictions from the 5-fold cross-validation setup. That gives a more honest view of the model's typical errors than a resubstitution score would. We summarize the false positives and false negatives and show the confusion matrix.

In [ ]:
from sklearn.metrics import classification_report

best_oof = {
    'Logistic Regression': logreg_oof,
    'Random Forest': rf_oof,
    'Gradient Boosting': gb_oof,
}[best_model_name]

cm_final = confusion_matrix(y_full, best_oof)
tn, fp, fn, tp = cm_final.ravel()

print(classification_report(y_full, best_oof, zero_division=0))
print(f'False positives: {fp}')
print(f'False negatives: {fn}')

plt.figure(figsize=(5, 4))
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Final Model Confusion Matrix: {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### 4.4 Feature Importance

To understand which engineered features matter most, we inspect feature importance. If the model exposes built-in importances, we use those directly. Otherwise, we compute permutation importance so the explanation still works for any model.

In [ ]:
from sklearn.inspection import permutation_importance

feature_names = X_full.columns

if hasattr(final_model, 'feature_importances_'):
    importances = pd.Series(final_model.feature_importances_, index=feature_names).sort_values(ascending=False)
    importance_title = f'{best_model_name} Feature Importance'
else:
    perm = permutation_importance(final_model, X_full, y_full, n_repeats=10, random_state=42, scoring='f1')
    importances = pd.Series(perm.importances_mean, index=feature_names).sort_values(ascending=False)
    importance_title = f'{best_model_name} Permutation Importance'

plt.figure(figsize=(10, 5))
sns.barplot(x=importances.values, y=importances.index, color='#4c78a8')
plt.title(importance_title)
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### 4.5 Summary

At this point we have selected the strongest model, retrained it on all labeled training data, and inspected both its error pattern and its key drivers. The notebook is now ready for the final Kaggle prediction step in the next section.

## 5. Final Prediction & Submission

In the last section we use the final model to predict the survival outcome for the Kaggle test set, preview the resulting submission table, and save the file in the required Kaggle format.

### 5.1 Generate Test Predictions

We apply the final trained model from Section 4 to the engineered test features. The output is a binary survival prediction for every passenger in the Kaggle test set.

In [ ]:
test_predictions = final_model.predict(X_test)
test_predictions = pd.Series(test_predictions, name='Survived')
test_predictions.value_counts().sort_index()

### 5.2 Submission Preview

Before saving the file, we preview the first rows of the submission table to verify that the structure is correct.

In [ ]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_predictions.astype(int)
})

submission.head()

### 5.3 Sanity Checks

We verify that the submission has the correct shape, the required columns, and only binary predictions.

In [ ]:
print(f'Submission shape: {submission.shape}')
print(submission.columns.tolist())
print(submission['Survived'].value_counts().sort_index())
print(f"Unique prediction values: {sorted(submission['Survived'].unique().tolist())}")

### 5.4 Save Submission File

The final submission is saved with a name that reflects the engineered features and model selection strategy.

In [ ]:
submission_file = 'submission_title_family_hascabin_cv.csv'
submission.to_csv(submission_file, index=False)
print(f'Saved {submission_file}')

### 5.5 Final Note

The notebook now contains a full end-to-end workflow: EDA, feature engineering, model comparison, final evaluation, and a Kaggle-ready submission file.